# 10.1 · Semantic search

Member 4 owns this method on `dev-semantic`. The reusable implementation is `discovery.rank.embeddings.EmbeddingRanker`.

MiniLM encodes text as 384 numbers. Normalizing each vector makes its dot product equal cosine similarity. FAISS `IndexFlatIP` finds the most similar vectors by checking the full index.

Both repository text and queries pass through the shared `tokenize()` before MiniLM encodes them. The five development queries drive text-format choices. Final test queries are reserved for Phase 14.

## 1. Load the corpus and shared tokenizer

Use the shared processed corpus and seed. Limit CPU threads for repeatable measurements on a laptop.

In [1]:
import hashlib
import json
from time import perf_counter

import faiss
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

from discovery.config import PATHS, set_seed
from discovery.preprocess.text import doc_text, tokenize
from discovery.rank.base import check_hits
from discovery.rank.devkit import DEV, load_corpus, measure, save_results, show_top
from discovery.rank.embeddings import MODEL, EmbeddingRanker

torch.set_num_threads(4)
faiss.omp_set_num_threads(4)
set_seed()

corpus = load_corpus()
ids = corpus["full_name"].tolist()
assert len(ids) == len(set(ids)) == 8_223
tech_tokens = tokenize("Built with Node.js and C#")
assert {"node.js", "c#"} <= set(tech_tokens)
print(f"{len(corpus):,} repositories | {tech_tokens}")
print("Development queries", *DEV, sep="\n  ")

8,223 repositories | ['built', 'node.js', 'c#']
Development queries
  beginner inventory management system nestjs postgresql
  point of sale app with react
  school management system laravel
  hotel booking system spring boot
  simple chat application flutter firebase


## 2. Build the FAISS prototype

Index one vector per repository. Start with metadata and the first 1,000 README characters so the metadata is available before MiniLM's 256-word-piece limit.

The local vector cache is keyed by the model, the normalized document text and the repository IDs in order. The first run encodes the corpus. The final benchmark measures a fresh build.

In [2]:
index_dir = PATHS["index"]
index_dir.mkdir(parents=True, exist_ok=True)

start = perf_counter()
prototype = SentenceTransformer(MODEL)
text = doc_text(corpus.assign(readme=corpus["readme"].fillna("").str.slice(0, 1000)))
text = [" ".join(tokenize(doc)) for doc in text]
cache_key = hashlib.sha256(json.dumps([MODEL, ids, text]).encode()).hexdigest()[:16]
vector_path = index_dir / f"semantic-prototype-{cache_key}.npy"

if vector_path.exists():
    vectors = np.load(vector_path)
    action = "Loaded"
else:
    vectors = prototype.encode(text, batch_size=32, normalize_embeddings=True)
    np.save(vector_path, vectors)
    action = "Built"

assert vectors.shape == (8_223, 384)
np.testing.assert_allclose(np.linalg.norm(vectors, axis=1), 1, atol=1e-6)
index = faiss.IndexFlatIP(vectors.shape[1])
index.add(vectors)
print(f"{action} {vectors.shape} vectors and index in {perf_counter() - start:.2f} s")

Could not reach the Hub (Cannot reach https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2: offline mode is enabled. To disable it, please unset the `HF_HUB_OFFLINE` environment variable.). Using cached commit hash for 'sentence-transformers/all-MiniLM-L6-v2'.


Built (8223, 384) vectors and index in 218.36 s


## 3. Search the prototype

Encode a development query with the same model and preprocessing, then map the ten nearest vector rows back to repository IDs.

In [3]:
query_tokens = " ".join(tokenize(DEV[0]))
query_vector = prototype.encode([query_tokens], normalize_embeddings=True)
similarities, rows = index.search(query_vector, 10)
baseline_hits = [
    {"doc_id": ids[row], "score": float(score)}
    for row, score in zip(rows[0], similarities[0])
]
check_hits(baseline_hits, 10)
for hit in baseline_hits:
    print(f"{hit['score']:.3f}  {hit['doc_id']}")
del prototype, vectors, index, text

0.787  jmw5598/inventory-management-system
0.727  shamim-42/Inventory-Management-System---Nestjs-Graphql
0.704  rohit-raj/inventory
0.699  shamim-42/Inventory-Management
0.666  samisams1/Inventory-Management-React-Nest-GraphQL-Prisma
0.666  fathiranugrahh/stockroom
0.651  TorgrimRL/inventory_x
0.647  vinothvino42/inventory-app
0.647  ninadnaik10/railway-booking-api
0.632  MrBartusek/stocked-up


## 4. Verify the reusable ranker

Check that the class matches the prototype and that its dense scores align with the input rows for hybrid search.

In [4]:
method = EmbeddingRanker()
speed = measure(lambda: method.fit(corpus), DEV)
hits = method.search(DEV[0], 10)
check_hits(hits, 10)
assert [h["doc_id"] for h in hits] == [h["doc_id"] for h in baseline_hits]
scores = method.scores(DEV[0])
assert scores.shape == (len(corpus),)
row_by_id = {doc_id: row for row, doc_id in enumerate(ids)}
np.testing.assert_allclose(
    [scores[row_by_id[h["doc_id"]]] for h in hits], [h["score"] for h in hits]
)
assert method.search("  ") == []
assert method.search(DEV[0], 0) == []
print("Prototype and ranker agree. One cosine score per corpus row.")
print(method.params())

Could not reach the Hub (Cannot reach https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2: offline mode is enabled. To disable it, please unset the `HF_HUB_OFFLINE` environment variable.). Using cached commit hash for 'sentence-transformers/all-MiniLM-L6-v2'.


Prototype and ranker agree. One cosine score per corpus row.
{'model': 'sentence-transformers/all-MiniLM-L6-v2', 'text': 'name + description + topics + README', 'max_chars_per_chunk': 1000, 'readme_chunks': 1, 'tokenizer': 'discovery.preprocess.text.tokenize'}


## 5. Check paraphrases and technology terms

Inspect three paraphrases and one exact-technology query as qualitative spot checks.

In [5]:
paraphrases = [
    "stock tracker for a small warehouse",
    "app to sell things in a shop",
    "manage students and classes",
    "nestjs postgresql",
]
show_top(method, corpus, paraphrases, k=5)


# stock tracker for a small warehouse   (5 hits)
    0.624  mobo-open-source/mobo_inventory              ★6      mobo Inventory for Odoo
    0.576  KushalVijay/Walmart-Sparkathon-Project-Ideas ★51     Walmart Sparkathon Projects for 2025
    0.566  Het161/Inventory                             ★7      Inventory management platform for OM Marketing Solutions — r
    0.542  rohanthanuj06-rgb/STOCK_MANAGEMENT_SYSTEM    ★10     not specified
    0.535  Prem-0007/Inventory_Management_System        ★18     not specified

# app to sell things in a shop   (5 hits)
    0.645  sairahul1526/shopy                           ★17     POS app
    0.611  codeforany/ebuy_e_commerce_app_development_f ★5      not specified
    0.601  nitinmehtaa/e-commerce-suit-in-flutter       ★107    A playground for your e-commerce app.
    0.587  arifhaidari/smart_shop_services              ★6      A cross-platform mobile application for the Smart Shop Point
    0.575  abuanwar072/Flutter-Shop-UI-Kit              ★576

## 6. Review the top ten development results

Inspect repository descriptions as well as names. A result can match the application domain while missing the requested framework.

In [6]:
show_top(method, corpus, DEV)


# beginner inventory management system nestjs postgresql   (10 hits)
    0.787  jmw5598/inventory-management-system          ★15     Inventory management system using Angular 9, Ngrx, NestJS, T
    0.727  shamim-42/Inventory-Management-System---Nest ★48     Inventory Management System Backend (Nestjs + Graphql)
    0.704  rohit-raj/inventory                          ★12     A simple inventory management system : Backend
    0.699  shamim-42/Inventory-Management               ★8      Inventory Management Backend developed with Nestjs, Graphql 
    0.666  samisams1/Inventory-Management-React-Nest-Gr ★13     Inventory Management System with React, TypeScript, NestJS, 
    0.666  fathiranugrahh/stockroom                     ★6      Full-stack inventory management for small businesses (UMKM).
    0.651  TorgrimRL/inventory_x                        ★5      Project for INF-2900 Software Engineering 
    0.647  vinothvino42/inventory-app                   ★5      An open-source inventory mana

**Two good results**

- `jmw5598/inventory-management-system` ranked first for the inventory query. Its description names NestJS and PostgreSQL, and its cleaned README is empty.
- `JohannesMilke/firebase_chat_example` ranked first for the chat query. Its description matches the application and names both Flutter and Firebase.

**Two misses**

- `ninadnaik10/railway-booking-api` ranked ninth for the inventory query. Its topics list NestJS and PostgreSQL, but it manages train reservations.
- `Mariusz94/BusTicketsBookingSpringBoot` ranked first for the hotel query. Its description matches Spring Boot and ?booking system?, but the project books bus tickets.

## 7. Compare text formats on development queries

Compare metadata alone, metadata with the README opening, and up to three README chunks. For chunks, the highest cosine for each repository is its score. The overlap counts measure how much rankings change. Use the displayed results to judge relevance until Phase 13 labels are available.

In [7]:
opening_results = [[h["doc_id"] for h in method.search(q, 10)] for q in DEV]
summary = [("README opening", speed["build_s"], "reference")]

for label, settings in [
    ("Metadata only", {"max_chars": 0}),
    ("README chunks", {"max_chars": 1000, "readme_chunks": 3}),
]:
    start = perf_counter()
    candidate = EmbeddingRanker(**settings).fit(corpus)
    build_s = perf_counter() - start
    print(f"\n{label}")
    show_top(candidate, corpus, DEV)
    overlap = [
        len(set(opening) & {h["doc_id"] for h in candidate.search(q, 10)})
        for q, opening in zip(DEV, opening_results)
    ]
    summary.append((label, build_s, ", ".join(map(str, overlap))))
    del candidate

print("\nFormat             Fit (s)  Top-10 overlap with README opening (five queries)")
for label, build_s, overlap in summary:
    print(f"{label:18} {build_s:7.2f}  {overlap}")

Could not reach the Hub (Cannot reach https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2: offline mode is enabled. To disable it, please unset the `HF_HUB_OFFLINE` environment variable.). Using cached commit hash for 'sentence-transformers/all-MiniLM-L6-v2'.



Metadata only

# beginner inventory management system nestjs postgresql   (10 hits)
    0.823  shamim-42/Inventory-Management-System---Nest ★48     Inventory Management System Backend (Nestjs + Graphql)
    0.803  shamim-42/Inventory-Management               ★8      Inventory Management Backend developed with Nestjs, Graphql 
    0.787  jmw5598/inventory-management-system          ★15     Inventory management system using Angular 9, Ngrx, NestJS, T
    0.777  samisams1/Inventory-Management-React-Nest-Gr ★13     Inventory Management System with React, TypeScript, NestJS, 
    0.756  invntio/easybooks                            ★5      Under construction. An open-source inventory management syst
    0.686  fathiranugrahh/stockroom                     ★6      Full-stack inventory management for small businesses (UMKM).
    0.684  MrBartusek/stocked-up                        ★13     Complete Inventory Management System that focuses on managin
    0.676  hmangam78/bookstoremanager         

Could not reach the Hub (Cannot reach https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2: offline mode is enabled. To disable it, please unset the `HF_HUB_OFFLINE` environment variable.). Using cached commit hash for 'sentence-transformers/all-MiniLM-L6-v2'.



README chunks

# beginner inventory management system nestjs postgresql   (10 hits)
    0.787  jmw5598/inventory-management-system          ★15     Inventory management system using Angular 9, Ngrx, NestJS, T
    0.762  shamim-42/Inventory-Management               ★8      Inventory Management Backend developed with Nestjs, Graphql 
    0.749  samisams1/Inventory-Management-React-Nest-Gr ★13     Inventory Management System with React, TypeScript, NestJS, 
    0.727  shamim-42/Inventory-Management-System---Nest ★48     Inventory Management System Backend (Nestjs + Graphql)
    0.704  rohit-raj/inventory                          ★12     A simple inventory management system : Backend
    0.700  ninadnaik10/railway-booking-api              ★22     Railway Booking system API
    0.686  invntio/easybooks                            ★5      Under construction. An open-source inventory management syst
    0.664  fathiranugrahh/stockroom                     ★6      Full-stack inventory managemen

## 8. Benchmark and save the selected setting

Report the fresh fit and median query time measured in step 4 with the model download already cached. The reported index size is the FAISS file size in decimal MB. The local vectors and repository row mapping are also saved for reuse. Relevance metrics remain blank until Phase 14. After labels and the evaluator land, run `evaluate_method(selected, split="test")`.

In [8]:
selected = method
index_path = index_dir / "semantic.faiss"
faiss.write_index(selected._index, str(index_path))
np.save(index_dir / "semantic-vectors.npy", selected._index.reconstruct_n())
(index_dir / "semantic-rows.json").write_text(
    json.dumps({"doc_ids": ids, "owners": selected._owners.tolist(), "params": selected.params()}),
    encoding="utf-8",
)
speed["index_mb"] = round(index_path.stat().st_size / 1e6, 1)
print(speed)
print(save_results(selected, speed).relative_to(PATHS["eval"].parent))

{'build_s': 243.34, 'query_ms_median': 8.64, 'index_mb': 12.6}
eval\results\semantic.json


## Findings

Keep metadata plus the first 1,000 README characters as the provisional default. It put the inventory app that explicitly uses NestJS and PostgreSQL first, up from third with metadata alone, and found the Spring Boot hotel backend at rank two. Metadata alone was the fastest to fit at 36.88 seconds. The three-chunk setting kept the same first result on four of five queries and took 584.67 seconds to fit, about 2.4 times as long as the README opening.

The paraphrases surfaced warehouse inventory, point-of-sale, and school-management projects. The hotel query exposed a recurring miss. All three formats placed bus booking first. The development rankings are a qualitative comparison until labelled NDCG@10 is available.